# MBX Node Annotation

Builds `Data/mbx_node_annotations.csv`, the metabolite-identification crosswalk consumed by
`pipeline.ipynb` (cell 9, MBX layer) and `02_seed_selection.py`'s metabolite tier. Standalone
and re-runnable: no local file in this project maps the 25 GLASSO-selected MBX feature IDs
(e.g. `C18n_QI7515`) to a compound name or mass -- this notebook sources that data externally
and documents exactly where from.

**METHOD**

The HMP2-supplied `Data/hmp2_mbx_metabolite_annotations.xlsx` only names 62 C18-negative-mode
compounds (keyed by bare `QIxxxx`, not `Method_QIxxxx`), and none of the 25 selected nodes are
among them; this metadata was never retained in this project's `mbx_metabolomics_raw.tsv`. Two
external, published resources are used instead:

1. **Official HMP2 feature metadata** (`HMP2_metabolomics_w_metadata.biom.gz`, IBDMDB products page,
   https://ibdmdb.org/downloads/html/products_MBX_2017-03-01.html) -- the merged BIOM file carries
   per-feature m/z, retention time, and (where known) name/HMDB ID for all ~82k features.
2. **Bhosle et al. (2024)**, *Molecular Systems Biology* 20(4):338-361, doi:10.1038/s44320-024-00027-8
   -- ran a covariance-based annotation method (MACARRoN) on this same iHMP/HMP2 dataset. Their
   Dataset EV3 assigns ~37k features to covariance modules; a feature that covaries with a named
   reference standard inherits a putative identity.

**LIMITATIONS**

A KEGG mass-based candidate search is also run as a secondary check, but is *not* a confirmed
identification -- see the caveats at the end of this notebook.

**RUN**

Both consuming scripts degrade gracefully if this file is absent, so this notebook can run any
time -- ideally before them, for full annotation coverage.

## 1. Full HMP2 feature metadata (m/z, RT, name, HMDB) — all ~82k features

In [1]:
import os, gzip, json as _json, urllib.request
import pandas as pd

BIOM_URL = "https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/products/HMP2/MBX/HMP2_metabolomics_w_metadata.biom.gz"
BIOM_GZ_PATH = "Data/Unprocessed_Other/HMP2_metabolomics_w_metadata.biom.gz"
FULL_METADATA_OUT = "Data/hmp2_mbx_full_feature_metadata.csv"

os.makedirs(os.path.dirname(BIOM_GZ_PATH), exist_ok=True)
if not os.path.exists(BIOM_GZ_PATH):
    print("Downloading BIOM file (~145MB)...")
    urllib.request.urlretrieve(BIOM_URL, BIOM_GZ_PATH)
print(f"{BIOM_GZ_PATH}: {os.path.getsize(BIOM_GZ_PATH)/1e6:.1f} MB")

with gzip.open(BIOM_GZ_PATH, "rt") as f:
    biom = _json.load(f)

rows = []
for r in biom["rows"]:
    m = r["metadata"]
    rows.append({
        "feature_id": r["id"],
        "method": m.get("Method", ""),
        "rt_min": m.get("RT", ""),
        "mz": m.get("m/z", ""),
        "hmp2_metabolite_name": m.get("Metabolite", ""),
        "hmp2_hmdb_id": m.get("HMDB (*Representative ID)", ""),
        "pooled_qc_cv": m.get("Pooled QC sample CV", ""),
    })
full_meta = pd.DataFrame(rows)
full_meta.to_csv(FULL_METADATA_OUT, index=False)
print(f"Saved {len(full_meta)} features to {FULL_METADATA_OUT}")
EMPTY = ""
print(f"Named (non-blank Metabolite): {(full_meta['hmp2_metabolite_name'] != EMPTY).sum()}")


Data/Unprocessed_Other/HMP2_metabolomics_w_metadata.biom.gz: 145.5 MB


Saved 81867 features to Data/hmp2_mbx_full_feature_metadata.csv
Named (non-blank Metabolite): 592


## 2. Bhosle et al. (2024) MACARRoN module assignments (Dataset EV3)

In [2]:
EV3_URL = "https://static-content.springer.com/esm/art%3A10.1038%2Fs44320-024-00027-8/MediaObjects/44320_2024_27_MOESM3_ESM.xlsx"
EV3_XLSX_PATH = "Data/macarron_ev3_module_assignments.xlsx"
EV3_CSV_PATH = "Data/macarron_ev3_module_assignments.csv"

if not os.path.exists(EV3_XLSX_PATH):
    print("Downloading Bhosle et al. 2024 Dataset EV3...")
    urllib.request.urlretrieve(EV3_URL, EV3_XLSX_PATH)
print(f"{EV3_XLSX_PATH}: {os.path.getsize(EV3_XLSX_PATH)/1e6:.2f} MB")

ev3 = pd.read_excel(EV3_XLSX_PATH, sheet_name="Module assignments")
ev3.to_csv(EV3_CSV_PATH, index=False)
print(f"Saved {len(ev3)} module-assigned features to {EV3_CSV_PATH}")

description = pd.read_excel(EV3_XLSX_PATH, sheet_name="Description")
if len(description.columns):
    print("\nDataset EV3 description (from source spreadsheet):")
    print(description.columns[0])


Data/macarron_ev3_module_assignments.xlsx: 1.20 MB


Saved 37201 module-assigned features to Data/macarron_ev3_module_assignments.csv



Dataset EV3 description (from source spreadsheet):
Of the 37,201 primary features, 35,594 clustered into 355 covariance modules(1607 singetons / module 0). Features that covaried with a standard metabolite are indicated by "Covaries with standard" = 1. Related classes indicates the chemical taxonomy of the standards in the same module as a feature. Metabolites including bile acid conjugates and triterpenoids that were not included as standards in the original HMP2 metabolomics datatset are indicated by ***.


## 3. The 25 GLASSO-selected MBX nodes: metadata + module assignment

In [3]:
selected_ids = pd.read_csv("Data/mbx_metabolomics_log_processed.csv", index_col=0).columns.tolist()
print(f"{len(selected_ids)} selected MBX network nodes")

node_meta = full_meta.set_index("feature_id").loc[selected_ids].reset_index()
node_meta.to_csv("Data/mbx_selected_node_metadata.csv", index=False)
node_meta.head()


25 selected MBX network nodes


,feature_id,method,rt_min,mz,hmp2_metabolite_name,hmp2_hmdb_id,pooled_qc_cv
0,C18n_QI10861,C18-neg,9.36,522.286,,,0.074553493
1,HILn_QI18128,HILIC-neg,4.27,152.0719,,,0.513094449
2,C18n_QI382,C18-neg,1.58,123.0076,,,0.051788873
3,C18n_QI7515,C18-neg,15.35,414.2655,,,0.098918174
4,HILn_QI2693,HILIC-neg,3.97,227.0676,,,0.694622241


## 4. Mass-based candidate search (KEGG, ±10 ppm)

**Not a confirmed identification.** This queries KEGG's ~19,000-compound database (far smaller than
PubChem or HMDB) for any compound whose mass falls within tolerance of each feature's neutral mass
(computed from m/z assuming the most common single-charge adduct per ionization mode: [M-H]⁻ for
negative modes, [M+H]⁺ for positive modes). A single-database mass match without retention-time or
MS/MS confirmation is Level 4/5 (MSI scale) at best — several of the hits found (e.g. an opioid, an
asthma drug, industrial epoxy resin components) are biologically implausible for stool/blood samples,
illustrating why mass alone does not discriminate between isobaric candidates. Kept as supplementary,
low-confidence context only; not used to relabel network nodes.

An automated PubChem PUG-REST mass-range search was also attempted and did not work: the modern REST
API has no GET-based numeric-range search, and the legacy PUG XML gateway (`pug.cgi`) is reachable
but requires an undocumented DTD-validated schema that could not be reliably reconstructed without
official docs access. Documented here so a future attempt does not re-spend the same time.

In [4]:
import time

PROTON = 1.007276

def neutral_mass(mz, method):
    mz = float(mz)
    return mz + PROTON if "neg" in method.lower() else mz - PROTON

def kegg_search(neutral, ppm=10):
    tol = neutral * ppm / 1e6
    lo, hi = neutral - tol, neutral + tol
    url = f"https://rest.kegg.jp/find/compound/{lo:.4f}-{hi:.4f}/exact_mass"
    with urllib.request.urlopen(url, timeout=20) as r:
        text = r.read().decode()
    return [tuple(l.split("\t")) for l in text.strip().split("\n") if l]

def kegg_names(cids):
    names = {}
    for i in range(0, len(cids), 10):
        batch = cids[i:i+10]
        url = "https://rest.kegg.jp/list/" + "+".join(batch)
        with urllib.request.urlopen(url, timeout=20) as r:
            text = r.read().decode()
        for line in text.strip().split("\n"):
            if not line:
                continue
            cid, rest = line.split("\t", 1)
            names[cid] = rest.split(";")[0]
        time.sleep(0.4)
    return names

kegg_rows = []
for _, row in node_meta.iterrows():
    fid, method, mz = row["feature_id"], row["method"], row["mz"]
    if not mz or pd.isna(mz):
        kegg_rows.append({**row, "neutral_mass": None, "assumed_adduct": None, "kegg_hits": 0, "kegg_names": ""})
        continue
    nm = neutral_mass(mz, method)
    adduct = "[M-H]-" if "neg" in method.lower() else "[M+H]+"
    hits = kegg_search(nm)
    cids = [h[0] for h in hits]
    names = kegg_names(cids)
    name_str = "; ".join(f"{c}:{names.get(c,'?')}" for c in cids)
    kegg_rows.append({**row, "neutral_mass": round(nm, 4), "assumed_adduct": adduct,
                       "kegg_hits": len(hits), "kegg_names": name_str})
    time.sleep(0.4)

kegg_df = pd.DataFrame(kegg_rows)
kegg_df.to_csv("Data/mbx_node_kegg_annotations.csv", index=False)
print(f"Saved KEGG candidate search for {len(kegg_df)} nodes")
kegg_df[["feature_id","mz","neutral_mass","kegg_hits","kegg_names"]]


Saved KEGG candidate search for 25 nodes


,feature_id,mz,neutral_mass,kegg_hits,kegg_names
0,C18n_QI10861,522.286,523.2933,0,
1,HILn_QI18128,152.0719,153.0792,3,C03758:Dopamine; C04227:Octopamine; C16666:Van...
2,C18n_QI382,123.0076,124.0149,3,"C07103:2-Hydroxy-1,4-benzoquinone; C20899:Fura..."
3,C18n_QI7515,414.2655,415.2728,1,C07241:Salmeterol
4,HILn_QI2693,227.0676,228.0749,1,C00526:Deoxyuridine
5,C18n_QI864,163.0391,164.0464,9,C00166:Phenylpyruvate; C00811:4-Coumarate; C01...
6,C18n_QI1309,193.0499,194.0572,10,C00779:Scytalone; C01494:Ferulate; C02379:6-Hy...
7,HILn_QI10329,122.0613,123.0686,2,C19191:ortho-Anisidine; C19326:para-Anisidine
8,C18n_QI11963,568.3306,569.3379,0,
9,HILn_QI3463,261.1344,262.1417,1,C19537:Triethylene glycol diglycidyl ether


## 5. Consolidated annotation crosswalk

In [5]:
ev3_lookup = ev3.rename(columns={"Feature": "feature_id"})[
    ["feature_id","Module","Covaries_with_standard","Anchor","Related_classes"]]

master = kegg_df.merge(ev3_lookup, on="feature_id", how="left")

def status(row):
    if row.get("Covaries_with_standard") == 1:
        return "putative (MACARRoN covariance with reference standard)"
    if pd.notna(row.get("kegg_hits")) and row["kegg_hits"] >= 1:
        return f"unconfirmed mass-consistent KEGG candidate(s), n={int(row['kegg_hits'])}"
    if pd.isna(row.get("Module")):
        return "unidentified (not covered by MACARRoN module analysis; no KEGG candidate)"
    return "unidentified (no covarying standard; no KEGG candidate)"

master["annotation_status"] = master.apply(status, axis=1)
master = master[["feature_id","method","rt_min","mz","neutral_mass","assumed_adduct",
                  "kegg_hits","kegg_names","Module","Covaries_with_standard","Anchor",
                  "Related_classes","annotation_status"]]
master.to_csv("Data/mbx_node_annotations.csv", index=False)

print(master["annotation_status"].value_counts().to_string())
master


annotation_status
unconfirmed mass-consistent KEGG candidate(s), n=1                           8
unidentified (no covarying standard; no KEGG candidate)                      6
putative (MACARRoN covariance with reference standard)                       6
unidentified (not covered by MACARRoN module analysis; no KEGG candidate)    2
unconfirmed mass-consistent KEGG candidate(s), n=3                           1
unconfirmed mass-consistent KEGG candidate(s), n=10                          1
unconfirmed mass-consistent KEGG candidate(s), n=2                           1


,feature_id,method,rt_min,mz,neutral_mass,assumed_adduct,kegg_hits,kegg_names,Module,Covaries_with_standard,Anchor,Related_classes,annotation_status
0,C18n_QI10861,C18-neg,9.36,522.286,523.2933,[M-H]-,0,,79.0,0.0,NaN,NaN,unidentified (no covarying standard; no KEGG c...
1,HILn_QI18128,HILIC-neg,4.27,152.0719,153.0792,[M-H]-,3,C03758:Dopamine; C04227:Octopamine; C16666:Van...,30.0,1.0,N-acetylhistamine,Organonitrogen compounds; Azoles; Carboxylic a...,putative (MACARRoN covariance with reference s...
2,C18n_QI382,C18-neg,1.58,123.0076,124.0149,[M-H]-,3,"C07103:2-Hydroxy-1,4-benzoquinone; C20899:Fura...",5.0,0.0,NaN,NaN,"unconfirmed mass-consistent KEGG candidate(s),..."
3,C18n_QI7515,C18-neg,15.35,414.2655,415.2728,[M-H]-,1,C07241:Salmeterol,99.0,0.0,NaN,NaN,"unconfirmed mass-consistent KEGG candidate(s),..."
4,HILn_QI2693,HILIC-neg,3.97,227.0676,228.0749,[M-H]-,1,C00526:Deoxyuridine,295.0,0.0,NaN,NaN,"unconfirmed mass-consistent KEGG candidate(s),..."
5,C18n_QI864,C18-neg,3.5,163.0391,164.0464,[M-H]-,9,C00166:Phenylpyruvate; C00811:4-Coumarate; C01...,7.0,1.0,malate,Hydroxy acids and derivatives; Carboxylic acid...,putative (MACARRoN covariance with reference s...
6,C18n_QI1309,C18-neg,3.57,193.0499,194.0572,[M-H]-,10,C00779:Scytalone; C01494:Ferulate; C02379:6-Hy...,5.0,0.0,NaN,NaN,"unconfirmed mass-consistent KEGG candidate(s),..."
7,HILn_QI10329,HILIC-neg,3.96,122.0613,123.0686,[M-H]-,2,C19191:ortho-Anisidine; C19326:para-Anisidine,88.0,1.0,homovanillate,Phenols,putative (MACARRoN covariance with reference s...
8,C18n_QI11963,C18-neg,12.23,568.3306,569.3379,[M-H]-,0,,125.0,0.0,NaN,NaN,unidentified (no covarying standard; no KEGG c...
9,HILn_QI3463,HILIC-neg,3.92,261.1344,262.1417,[M-H]-,1,C19537:Triethylene glycol diglycidyl ether,0.0,0.0,NaN,NaN,"unconfirmed mass-consistent KEGG candidate(s),..."
